# Simulation of PSI relaxation kinetics with mutants

In [ ]:
%matplotlib inline

# Import packages and functions
import modelbase
import numpy as np
import pandas as pd
import os
import importlib
import sys
import re
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, PercentFormatter
from matplotlib.patches import Rectangle
from matplotlib.colors import TwoSlopeNorm, CenteredNorm, SymLogNorm, Normalize
import matplotlib.colors as colors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patches as mpatches

from PIL import Image

from scipy.signal import find_peaks, savgol_filter
from scipy.stats import iqr
from scipy.integrate import simpson
from scipy.optimize import minimize

from modelbase.ode import Model, Simulator, mca
from modelbase.ode import ratelaws as rl
from modelbase.ode import ratefunctions as rf

from concurrent.futures import ProcessPoolExecutor
from functools import partial
from pathlib import Path
from sympy import Matrix, lambdify, linsolve, symbols
from warnings import warn
from os import listdir
from os.path import join
from functools import reduce
from operator import mul

# Helper functions
sys.path.append("../Code")
import functions as fnc
import calculate_parameters_restruct as prm
import functions_light_absorption as lip

# Import model functions
from get_current_model import get_model
from module_update_phycobilisomes import add_OCP

idx = pd.IndexSlice


from functions_custom_steady_state_simulator import simulate_to_steady_state_custom, _find_steady_state, get_response_coefficients, get_response_coefficients_array, get_response_coefficients_df, calculate_ss_Q_red, get_steadystate_y0
from function_residuals import residual_normalisation
from functions_fluorescence_simulation import make_lights, make_adjusted_lights, create_protocol_NPQ, create_protocol_NPQ_short, create_protocol_noNPQ

## Simulate default behaviour

In [ ]:
continuous_light = lip.light_gaussianLED(670, 20) # FR
MT_light = lip.light_gaussianLED(440, 5000) # MT: Blue
# after_light = lip.light_gaussianLED(630, 20)
after_light = lip.light_gaussianLED(670, 20)

In [ ]:
# Create the protocol
protocol = fnc.create_protocol_const(
    continuous_light, 0.1, None
)

protocol = fnc.create_protocol_const(
   MT_light , 0.1, protocol
)

protocol = fnc.create_protocol_const(
    after_light, 1, protocol
)

In [ ]:
# Default model
m0 = get_model(check_consistency=False, verbose=False, get_y0=False)

# Increased CET
mCET = get_model(check_consistency=False, verbose=False, get_y0=False)
mCET.update_parameter("vNQ_max", mCET.get_parameter("vNQ_max") * 2)

models = {
    "default": m0,
    "highCET": mCET
}

In [ ]:
fig,ax= plt.subplots(figsize=(10,7))

for model_name, m in models.items():
    # Get the simulator
    y0_ss = get_steadystate_y0(m, y0, continuous_light)
    s = Simulator(m)
    s.initialise(y0_ss)

    s = fnc.simulate_protocol(s, protocol, retry_unsuccessful=True)
    # Plot
    P700 = s.get_full_results_df().loc[:,"Y2"] / m.get_parameter("PSItot")

    ax.plot(P700, label=model_name)

ax.set_ylabel("Fraction of P700$^+$ [rel.]")
ax.set_xlabel("Time [s]")
ax.legend(title="Model")
fnc.add_lightbar(s, ax, 1000, color="mono", remove_pulses=False, scale="log", size=0.06, time_offset=0, )